In [0]:
import requests
from pyspark.sql import functions as F

destinations = ["Kyoto", "Paris", "Reykjavik", "Marrakesh", "Vancouver"]

geocode_results = []
skipped = []

for dest in destinations:
    resp = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": dest, "count": 1},
        timeout=10
    )
    data = resp.json()

    if "results" not in data or not data["results"]:
        skipped.append(dest)
        continue

    result = data["results"][0]
    geocode_results.append({
        "destination": dest,
        "lat": result["latitude"],
        "lon": result["longitude"],
        "country": result.get("country"),
        "timezone": result.get("timezone"),
    })

if skipped:
    print(f"⚠️ No geocoding match found for: {skipped}")

geocode_df = spark.createDataFrame(geocode_results)
geocode_df = geocode_df.withColumn("ingested_at", F.current_timestamp())
geocode_df.write.mode("overwrite").saveAsTable("trip_planner.bronze.destinations_geocoded")
geocode_df.show(truncate=False)

In [0]:
weather_results = []
skipped_weather = []

for row in geocode_results:
    try:
        weather_resp = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": row["lat"], "longitude": row["lon"],
                "current_weather": "true",
                "hourly": "temperature_2m,precipitation_probability",
            },
            timeout=10
        )
        weather_data = weather_resp.json()

        air_resp = requests.get(
            "https://air-quality-api.open-meteo.com/v1/air-quality",
            params={
                "latitude": row["lat"], "longitude": row["lon"],
                "current": "pm10,uv_index",
            },
            timeout=10
        )
        air_data = air_resp.json()

        weather_results.append({
            "destination": row["destination"],
            "temperature_c": weather_data["current_weather"]["temperature"],
            "windspeed": weather_data["current_weather"]["windspeed"],
            "weather_code": weather_data["current_weather"]["weathercode"],
            "pm10": air_data["current"].get("pm10"),
            "uv_index": air_data["current"].get("uv_index"),
        })
    except (KeyError, requests.RequestException) as e:
        skipped_weather.append((row["destination"], str(e)))

if skipped_weather:
    print(f"⚠️ Skipped: {skipped_weather}")

weather_df = spark.createDataFrame(weather_results)
weather_df = weather_df.withColumn("ingested_at", F.current_timestamp())
weather_df.write.mode("overwrite").saveAsTable("trip_planner.bronze.weather_snapshots")
weather_df.show(truncate=False)

In [0]:
weather_results = []
skipped_weather = []

for row in geocode_results:
    try:
        weather_resp = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": row["lat"], "longitude": row["lon"],
                "current_weather": "true",
                "hourly": "temperature_2m,precipitation_probability",
            },
            timeout=10
        )
        weather_data = weather_resp.json()

        air_resp = requests.get(
            "https://air-quality-api.open-meteo.com/v1/air-quality",
            params={
                "latitude": row["lat"], "longitude": row["lon"],
                "current": "pm10,uv_index",
            },
            timeout=10
        )
        air_data = air_resp.json()

        pm10 = air_data["current"].get("pm10")
        uv = air_data["current"].get("uv_index")

        weather_results.append({
            "destination": row["destination"],
            "temperature_c": float(weather_data["current_weather"]["temperature"]),
            "windspeed": float(weather_data["current_weather"]["windspeed"]),
            "weather_code": int(weather_data["current_weather"]["weathercode"]),
            "pm10": float(pm10) if pm10 is not None else None,
            "uv_index": float(uv) if uv is not None else None,
        })
    except (KeyError, requests.RequestException) as e:
        skipped_weather.append((row["destination"], str(e)))

if skipped_weather:
    print(f"⚠️ Skipped: {skipped_weather}")

weather_df = spark.createDataFrame(weather_results)
weather_df = weather_df.withColumn("ingested_at", F.current_timestamp())
weather_df.write.mode("overwrite").saveAsTable("trip_planner.bronze.weather_snapshots")
weather_df.show(truncate=False)

In [0]:
GEONAMES_USERNAME = "eighty8kane"

wiki_results = []
skipped_wiki = []

for dest in destinations:
    try:
        resp = requests.get(
            "http://api.geonames.org/wikipediaSearchJSON",
            params={"q": dest, "maxRows": 5, "username": GEONAMES_USERNAME},
            timeout=10
        )
        data = resp.json()

        entries = data.get("geonames", [])
        if not entries:
            skipped_wiki.append(dest)
            continue

        for entry in entries:
            wiki_results.append({
                "destination": dest,
                "title": entry.get("title"),
                "summary": entry.get("summary"),
                "lat": float(entry.get("lat")) if entry.get("lat") is not None else None,
                "lon": float(entry.get("lng")) if entry.get("lng") is not None else None,
                "wikipedia_url": entry.get("wikipediaUrl"),
                "feature": entry.get("feature"),
            })
    except (KeyError, requests.RequestException) as e:
        skipped_wiki.append((dest, str(e)))

if skipped_wiki:
    print(f"⚠️ Skipped: {skipped_wiki}")

wiki_df = spark.createDataFrame(wiki_results)
wiki_df = wiki_df.withColumn("ingested_at", F.current_timestamp())
wiki_df.write.mode("overwrite").saveAsTable("trip_planner.bronze.destination_descriptions")
wiki_df.select("destination", "title", "summary").show(truncate=60)